# Part 14, notebook 5 of 6: the same 17 questions through Deep Agents and Google ADK

**The kitchen picture.** My harness is cooking at home from a short recipe: I chose every pan. **Deep Agents** is a meal kit: its own pans, a recipe book (a long built-in system prompt), a timer for big dishes (offloading above 20,000 tokens) and a sous-chef you can call (the `task` sub-agent tool). **ADK** is a professional kitchen with the stove and the sous-chef system included, but the pans are mostly left to you, plus a drawer of "environment" pans still labelled experimental.

Same local model, same questions, same scoring, 40-call cap for both frameworks (14 for my harness). This notebook **reads the saved runs**: the framework runs need their own venv (`.venv-adk`) and took hours. All numbers come from `ctx_frameworks.py`.

In [1]:
import sys, json, os
from pathlib import Path
os.environ.setdefault("AGENTIC_ZERO_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves (as in the post)
os.environ.setdefault("RUST_LOG", "error"); os.environ.setdefault("LANCE_LOG", "error")   # silence LanceDB deprecation chatter
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))                # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
import ctx_frameworks as fw
res = fw.collect()
print(list(res)[:6])

['harness (best set-up per task)', 'Deep Agents', 'Deep Agents + chunk sub-agent', 'ADK, my tools', 'ADK, my tools + chunk sub-agent', 'ADK, its own toolset (sandboxed)']


## The result table

In [2]:
print(f"{'set-up':36s} {'T1 /6':>6s} {'T2 /3':>6s} {'T3 /2':>6s} {'T4 /3':>6s} {'T5 /3':>6s} {'TOTAL /17':>10s} {'cap':>4s} {'overflow':>9s}")
for name in fw.SETS:
    r = res[name]; p = r["per_task"]
    print(f"{name:36s} " + " ".join(f"{p[t]['correct']:6d}" for t in fw.QN) + f" {r['correct']:10d} {r['cap']:4d} {r['overflow']:9d}")
print("\npasting the reports in (stuff):", {t: f"{v['correct']}/{v['n']}" for t, v in res["stuff"].items()})

set-up                                T1 /6  T2 /3  T3 /2  T4 /3  T5 /3  TOTAL /17  cap  overflow
harness (best set-up per task)            6      0      1      2      3         12    5         0
Deep Agents                               4      0      1      0      1          6   11         0
Deep Agents + chunk sub-agent             5      1      1      0      3         10    7         0
ADK, my tools                             6      0      0      1      2          9    7         1
ADK, my tools + chunk sub-agent           5      0      1      1      1          8    5         3
ADK, its own toolset (sandboxed)          6      1      2      1      2         12    5         0

pasting the reports in (stuff): {'T1': '0/6', 'T2': '3/3', 'T3': '2/2', 'T4': '3/3'}


**Read this as "no framework is clearly ahead", not as a ranking.** A difference of one or two runs is noise at 17 questions and one run each. The harness row uses the best set-up *per task* (named in `HARNESS_BEST`), which flatters it, and its cap was 14 calls against 40.

In [3]:
print("which set-up my harness row uses per task:", fw.HARNESS_BEST)

which set-up my harness row uses per task: {'T1': 'harness', 'T2': 'harness', 'T3': 'harness', 'T4': 'search-paths', 'T5': 'v-harness'}


## What a correct answer cost

Prompt tokens summed over all steps, per correct needle (T1) run.

In [4]:
rows = sorted(((res[n]["per_task"]["T1"]["tokens_per_correct"], n) for n in fw.SETS if res[n]["per_task"]["T1"]["tokens_per_correct"]))
for tok, n in rows: print(f"{tok:8,d}  {n}")

   2,834  harness (best set-up per task)
   5,056  ADK, my tools
   8,137  ADK, its own toolset (sandboxed)
  11,132  Deep Agents + chunk sub-agent
  13,099  Deep Agents
  40,197  ADK, my tools + chunk sub-agent


## The runs that failed mostly never answered

Split every failure into: wrong answer, hit the call cap, overflowed the window.

In [5]:
print(f"{'set-up':36s} {'ok':>4s} {'wrong':>6s} {'cap':>4s} {'overflow':>9s}")
for name in fw.SETS:
    r = res[name]; p = r["per_task"]
    print(f"{name:36s} {r['correct']:4d} {sum(x['wrong'] for x in p.values()):6d} {r['cap']:4d} {r['overflow']:9d}")

set-up                                 ok  wrong  cap  overflow
harness (best set-up per task)         12      0    5         0
Deep Agents                             6      0   11         0
Deep Agents + chunk sub-agent          10      0    7         0
ADK, my tools                           9      0    7         1
ADK, my tools + chunk sub-agent         8      1    5         3
ADK, its own toolset (sandboxed)       12      0    5         0


Deep Agents hit the cap on 18 of its 34 runs and ADK with my tools on 12 (plus 4 window overflows). Most failures are runs that **never gave an answer**, not wrong answers. (An earlier chat message of mine said 22/10/3. That was wrong; the saved files say 18/12/4, and the post was corrected.)

## How each framework is wired

The agent definitions, straight from the files. Deep Agents: `create_deep_agent` with a filesystem backend and a call limit. ADK: `LlmAgent` with plain Python functions as tools (type hints and the docstring become the schema), `AgentTool` for the sub-agent, and `RunConfig(max_llm_calls=40)`.

In [6]:
import re
src = (OUT.parent / "ctx_deepagents.py").read_text()
print(src[src.index("def build_agent"):src.index("def run(")])

def build_agent(arm: str, root: Path):
    return create_deep_agent(
        model=make_llm(), tools=[search_reports, similar_cases], system_prompt=SYSTEM,
        subagents=[CHUNK_ANALYST] if arm == "da+chunk" else None,
        backend=FilesystemBackend(root_dir=root, virtual_mode=True),
        middleware=[ModelCallLimitMiddleware(run_limit=CALL_LIMIT, exit_behavior="end")])





In [7]:
src = (OUT.parent / "ctx_adk.py").read_text()
i = src.index("TOOLS = ") if "TOOLS = " in src else 0
print(src[max(0, i - 600):i + 500])

th offset."""
    return h.t_read_file(path, offset, limit, paged=True)


def search_reports(query: str) -> str:
    """Semantic (by-meaning) search over the inspection reports. Returns the 4 best matches as file paths."""
    return h.t_search_reports(query, paths=True)


def similar_cases(image_id: str, k: int = 4) -> str:
    """Visual search: the k test images that look most like the given one (DINOv3 embeddings).
    Returns file paths under cases/, one record per image."""
    from ctx_visual import neighbours
    return "\n".join(f"cases/{i}.md" for i in neighbours(image_id, int(k)))


TOOLS = [as_text_error(f) for f in (ls, glob, grep, read_file, search_reports, similar_cases)]


def build_agent(arm: str, usage: Usage) -> LlmAgent:
    tools = list(TOOLS)
    if arm == "adk+chunk":
        analyst = LlmAgent(name="chunk_analyst", model=make_model(), instruction=h.ANALYST_SYSTEM, tools=[read_file],
                           description="Reads ONE file and returns only the facts

**Not covered:** this ran on one local model (Qwen3.8-27B), **not Gemini**. ADK is built around Gemini and I did not test it. A hosted model with a longer window and stronger tool use could behave very differently. The takeaways are about how to *measure*, not which framework to buy.